In [3]:
import pandas as pd
import duckdb

data = [
    # Device A：正常增长、下降
    ["A", "2026-01-15", 10],
    ["A", "2026-02-15", 15],
    ["A", "2026-03-15", 12],
    ["A", "2026-04-15", 18],

    # Device B：中间缺少上个月同日
    ["B", "2026-01-10", 5],
    ["B", "2026-03-10", 9],
    ["B", "2026-04-10", 6],

    # Device C：上个月值为 0，需要处理除零问题
    ["C", "2026-01-20", 0],
    ["C", "2026-02-20", 8],
    ["C", "2026-03-20", 4],

    # Device D：当前值为 0，增长率可以正常计算为 -100%
    ["D", "2026-01-05", 30],
    ["D", "2026-02-05", 0],
    ["D", "2026-03-05", 10],

    # Device E：连续为 0，增长率也不能简单除以 0
    ["E", "2026-01-01", 0],
    ["E", "2026-02-01", 0],
    ["E", "2026-03-01", 5],

    # Device F：只有一条记录，没有上个月数据
    ["F", "2026-03-12", 7],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "stat_date", "alarm_count"]
)

df["stat_date"] = pd.to_datetime(df["stat_date"])

print(df)



   device_id  stat_date  alarm_count
0          A 2026-01-15           10
1          A 2026-02-15           15
2          A 2026-03-15           12
3          A 2026-04-15           18
4          B 2026-01-10            5
5          B 2026-03-10            9
6          B 2026-04-10            6
7          C 2026-01-20            0
8          C 2026-02-20            8
9          C 2026-03-20            4
10         D 2026-01-05           30
11         D 2026-02-05            0
12         D 2026-03-05           10
13         E 2026-01-01            0
14         E 2026-02-01            0
15         E 2026-03-01            5
16         F 2026-03-12            7


## 题目要求

* **分别使用 SQL 和 Pandas 完成：**

- 计算每个设备当天报警次数，与上个月同日相比的：

    - 1. 上个月同日报警次数
    - 2. 变化量
    - 3. 增长率
    - 4. 增长率百分比

* **这里的上个月同日是：**

- 同一个 `device_id`
- 并且日期是当前日期的上一个自然月同一天

**例如：**

- 2026-02-15 对比 2026-01-15
- 2026-03-15 对比 2026-02-15
- 2026-04-10 对比 2026-03-10

### 最终输出字段

- `device_id`
- `stat_date`
- `alarm_count`
- `previous_month_alarm_count`
- `alarm_count_diff`
- `growth_rate`
- `growth_rate_pct`

In [17]:
query = """

WITH previous_table AS (
    SELECT
        device_id,
        stat_date + INTERVAL 1 MONTH AS stat_date,
        alarm_count AS previous_month_alarm_count
    FROM df
),

compare_table AS (
    SELECT
        curr.device_id,
        curr.stat_date,
        curr.alarm_count,
        prev.previous_month_alarm_count,
        curr.alarm_count - prev.previous_month_alarm_count AS alarm_count_diff
    FROM df AS curr
    LEFT JOIN previous_table AS prev
        ON curr.device_id = prev.device_id
       AND curr.stat_date = prev.stat_date
)

SELECT
    device_id,
    stat_date,
    alarm_count,
    previous_month_alarm_count,
    alarm_count_diff,
    CASE
        WHEN previous_month_alarm_count IS NULL
          OR previous_month_alarm_count = 0
        THEN NULL
        ELSE ROUND(alarm_count_diff * 1.0 / previous_month_alarm_count, 2)
    END AS growth_rate,
    CASE
        WHEN previous_month_alarm_count IS NULL
          OR previous_month_alarm_count = 0
        THEN NULL
        ELSE ROUND(alarm_count_diff * 100.0 / previous_month_alarm_count, 2)
    END AS growth_rate_pct
FROM compare_table
ORDER BY device_id, stat_date

"""

df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,stat_date,alarm_count,previous_month_alarm_count,alarm_count_diff,growth_rate,growth_rate_pct
0,A,2026-01-15,10,<NA>,<NA>,NaN,NaN
1,A,2026-02-15,15,10,5,0.50,50.00
2,A,2026-03-15,12,15,-3,-0.20,-20.00
3,A,2026-04-15,18,12,6,0.50,50.00
4,B,2026-01-10,5,<NA>,<NA>,NaN,NaN
5,B,2026-03-10,9,<NA>,<NA>,NaN,NaN
6,B,2026-04-10,6,9,-3,-0.33,-33.33
7,C,2026-01-20,0,<NA>,<NA>,NaN,NaN
8,C,2026-02-20,8,0,8,NaN,NaN
9,C,2026-03-20,4,8,-4,-0.50,-50.00


In [ ]:
df_prev_month = (
    df
    .assign(
        stat_date=lambda x: (
            x['stat_date'] + pd.DateOffset(months=1)
        )
    )
    .rename(
        columns={'alarm_count': 'previous_month_alarm_count'}
    )
)

df_pd = (
    df
    .merge(
        df_prev_month[
            ['device_id', 'stat_date', 'previous_month_alarm_count']
        ],
        on=['device_id', 'stat_date'],
        how='left'
    )
    .assign(
        alarm_count_diff=lambda x: (
            x['alarm_count'] - x['previous_month_alarm_count']
        ),
        growth_rate=lambda x: (
            (x['alarm_count_diff'] / x['previous_month_alarm_count'])
            .where(
                x['previous_month_alarm_count'].notna()
                & (x['previous_month_alarm_count'] != 0)
            )
        ),
        growth_rate_pct=lambda x: (
            (x['alarm_count_diff'] * 100 / x['previous_month_alarm_count'])
            .where(
                x['previous_month_alarm_count'].notna()
                & (x['previous_month_alarm_count'] != 0)
            )
        )
    )
    .sort_values(by=['device_id', 'stat_date'])
    .reset_index(drop=True)
)

df_pd

,device_id,stat_date,alarm_count,previous_month_alarm_count,alarm_count_diff,growth_rate,growth_rage_pct
0,A,2026-02-15,15,10.0,5.0,0.500000,50.000000
1,A,2026-03-15,12,15.0,-3.0,-0.200000,-20.000000
2,A,2026-04-15,18,12.0,6.0,0.500000,50.000000
3,B,2026-04-10,6,9.0,-3.0,-0.333333,-33.333333
4,C,2026-03-20,4,8.0,-4.0,-0.500000,-50.000000
5,D,2026-02-05,0,30.0,-30.0,-1.000000,-100.000000
